# Chapter 13: LangGraph 기초 (LangGraph Fundamentals)

LangGraph의 기본 개념을 단계적으로 실습합니다.

**Sections:**
- 13.0 Setup & Environment
- 13.1 Your First Graph (StateGraph, Node, Edge)
- 13.2 Graph State (상태 읽기/수정)
- 13.4 Multiple Schemas (입력/출력/내부 스키마 분리)
- 13.5 Reducer Functions (상태 누적)
- 13.6 Node Caching (CachePolicy)
- 13.7 Conditional Edges (조건부 분기)
- 13.8 Send API (동적 병렬 처리)
- 13.9 Command (노드 내부 라우팅)

---
## 13.0 Setup & Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

In [ ]:
# 패키지 확인
import langgraph
import langchain

print(f"langgraph: {langgraph.__version__}")
print(f"langchain: {langchain.__version__}")

---
## 13.1 Your First Graph

LangGraph의 세 가지 핵심 요소:
1. **State** — 그래프 전체에서 공유되는 데이터 (`TypedDict`)
2. **Node** — 상태를 입력으로 받는 개별 함수
3. **Edge** — 노드 간의 연결 (실행 순서)

특수 노드: `START` (시작점), `END` (종료점)

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


# 1단계: 상태 정의
class State(TypedDict):
    hello: str


graph_builder = StateGraph(State)


# 2단계: 노드 함수 정의
def node_one(state: State):
    print("node_one")


def node_two(state: State):
    print("node_two")


def node_three(state: State):
    print("node_three")


# 3단계: 그래프 구성
graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

# 4단계: 컴파일 & 실행
graph = graph_builder.compile()
result = graph.invoke({"hello": "world"})
print("\nResult:", result)

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 13.1

1. `START`를 생략하면 어떤 에러가 발생하는가?
2. 노드 순서를 변경하여 실행 흐름이 어떻게 달라지는지 관찰하라.
3. `add_node`에서 문자열 이름을 생략하면 함수 이름이 자동으로 사용되는가?

In [ ]:
# Try it yourself


---
## 13.2 Graph State — 상태 읽기와 수정

각 노드는:
1. 현재 상태를 **입력**으로 받는다
2. 딕셔너리를 **반환**하여 상태를 업데이트한다
3. 반환된 값은 기존 상태에 **덮어쓰기(overwrite)** 된다 (기본 동작)

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class State(TypedDict):
    hello: str
    a: bool


graph_builder = StateGraph(State)


def node_one(state: State):
    print("node_one", state)
    return {
        "hello": "from node one.",
        "a": True,
    }


def node_two(state: State):
    print("node_two", state)
    return {"hello": "from node two."}


def node_three(state: State):
    print("node_three", state)
    return {"hello": "from node three."}


graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

graph = graph_builder.compile()
result = graph.invoke({"hello": "world"})

print("\nFinal result:", result)

### Exercise 13.2

1. 초기 입력에 `a` 값을 포함하여 전달해 보라. 노드에서 어떻게 보이는가?
2. 노드에서 `State`에 없는 키를 반환하면 어떻게 되는가?
3. `node_two`에서 `a`를 `False`로 변경하면 `node_three`에서 어떤 값이 보이는가?

In [ ]:
# Try it yourself


---
## 13.4 Multiple Schemas — 다중 스키마

하나의 그래프에서 **입력/출력/내부** 스키마를 분리:

| 매개변수 | 역할 |
|----------|------|
| 첫 번째 인자 | 내부 전체 상태 (Private State) |
| `input_schema` | 외부 입력 형태 |
| `output_schema` | 외부 출력 형태 |

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class PrivateState(TypedDict):
    a: int
    b: int


class InputState(TypedDict):
    hello: str


class OutputState(TypedDict):
    bye: str


class MegaPrivate(TypedDict):
    secret: bool


graph_builder = StateGraph(
    PrivateState,
    input_schema=InputState,
    output_schema=OutputState,
)


def node_one(state: InputState) -> InputState:
    print("node_one ->", state)
    return {"hello": "world"}


def node_two(state: PrivateState) -> PrivateState:
    print("node_two ->", state)
    return {"a": 1}


def node_three(state: PrivateState) -> PrivateState:
    print("node_three ->", state)
    return {"b": 1}


def node_four(state: PrivateState) -> OutputState:
    print("node_four ->", state)
    return {"bye": "world"}


def node_five(state: OutputState):
    return {"secret": True}


def node_six(state: MegaPrivate):
    print(state)


graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)
graph_builder.add_node("node_four", node_four)
graph_builder.add_node("node_five", node_five)
graph_builder.add_node("node_six", node_six)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", "node_four")
graph_builder.add_edge("node_four", "node_five")
graph_builder.add_edge("node_five", "node_six")
graph_builder.add_edge("node_six", END)

graph = graph_builder.compile()
result = graph.invoke({"hello": "world"})

print("\nFinal result:", result)

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 13.4

1. `output_schema`를 지정하지 않으면 반환값이 어떻게 달라지는가?
2. `input_schema`에 없는 필드를 `invoke()`에 전달하면 어떻게 되는가?
3. 스키마 분리가 실제 프로덕션 환경에서 왜 중요한지 생각해 보라 (보안, API 설계).

In [ ]:
# Try it yourself


---
## 13.5 Reducer Functions — 리듀서

기본 "덮어쓰기" 대신 **리듀서(Reducer)**로 상태를 **누적**:

```python
Annotated[타입, 리듀서_함수]
```

리듀서 함수: `(old, new) -> 최종값`

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import Annotated
import operator


# 커스텀 리듀서 (operator.add와 동일한 동작)
def update_function(old, new):
    return old + new


class State(TypedDict):
    # messages: Annotated[list[str], update_function]
    messages: Annotated[list[str], operator.add]


graph_builder = StateGraph(State)


def node_one(state: State):
    return {
        "messages": ["Hello, nice to meet you!"],
    }


def node_two(state: State):
    return {}


def node_three(state: State):
    return {}


graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

graph = graph_builder.compile()
result = graph.invoke({"messages": ["Hello!"]})

print("Result:", result)

### Exercise 13.5

1. `node_two`에서도 메시지를 추가하여 누적이 정상 동작하는지 확인하라.
2. 리듀서 없이 동일한 코드를 실행하면 결과가 어떻게 달라지는가?
3. 커스텀 리듀서를 만들어 보라 (예: 중복 제거, 최대값만 유지).

In [ ]:
# Try it yourself


---
## 13.6 Node Caching

`CachePolicy(ttl=초)` + `InMemoryCache()`로 노드별 캐시 적용.

비용이 높은 연산(API 호출, LLM 호출)에서 성능을 크게 개선할 수 있다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache
from datetime import datetime
import time


class State(TypedDict):
    time: str


graph_builder = StateGraph(State)


def node_one(state: State):
    return {}


def node_two(state: State):
    return {"time": f"{datetime.now()}"}


def node_three(state: State):
    return {}


graph_builder.add_node("node_one", node_one)
graph_builder.add_node(
    "node_two",
    node_two,
    cache_policy=CachePolicy(ttl=20),  # 20초 캐시
)
graph_builder.add_node("node_three", node_three)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

graph = graph_builder.compile(cache=InMemoryCache())

In [ ]:
# 5초 간격으로 6번 실행 — 20초 이후 캐시가 만료되는지 관찰
for i in range(6):
    result = graph.invoke({})
    print(f"Run {i+1}: {result['time']}")
    if i < 5:
        time.sleep(5)

### Exercise 13.6

1. `ttl` 값을 `10`으로 변경하며 캐시 만료 타이밍을 관찰하라.
2. 캐시가 유용한 실제 시나리오를 생각해 보라 (API 요율 제한, 비용 절감).
3. 캐시가 문제가 될 수 있는 경우는? (실시간 데이터가 필요한 경우)

In [ ]:
# Try it yourself


---
## 13.7 Conditional Edges — 조건부 엣지

상태에 따라 **동적으로 다음 노드를 선택**:

```python
add_conditional_edges(
    출발_노드,
    분기_함수,
    매핑_딕셔너리  # {반환값: 목적지_노드}
)
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import Literal


class State(TypedDict):
    seed: int


graph_builder = StateGraph(State)


def node_one(state: State):
    print("node_one ->", state)
    return {}


def node_two(state: State):
    print("node_two ->", state)
    return {}


def node_three(state: State):
    print("node_three ->", state)
    return {}


def node_four(state: State):
    print("node_four ->", state)
    return {}


# 분기 함수: True/False 반환 → 매핑으로 노드 결정
def decide_path(state: State):
    return state["seed"] % 2 == 0


graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)
graph_builder.add_node("node_four", node_four)

# START에서 조건부 분기
graph_builder.add_conditional_edges(
    START,
    decide_path,
    {
        True: "node_one",   # seed 짝수 → node_one
        False: "node_two",  # seed 홀수 → node_two
    },
)

graph_builder.add_edge("node_one", "node_two")

# node_two에서도 조건부 분기
graph_builder.add_conditional_edges(
    "node_two",
    decide_path,
    {
        True: "node_three",
        False: "node_four",
    },
)

graph_builder.add_edge("node_three", END)
graph_builder.add_edge("node_four", END)

graph = graph_builder.compile()

In [ ]:
# 짝수 seed로 실행
print("=== seed=42 (짝수) ===")
result = graph.invoke({"seed": 42})
print("Result:", result)

In [ ]:
# 홀수 seed로 실행
print("=== seed=7 (홀수) ===")
result = graph.invoke({"seed": 7})
print("Result:", result)

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 13.7

1. `seed` 값을 다양하게 변경하며 실행 경로가 어떻게 달라지는지 관찰하라.
2. 노드 이름을 직접 반환하는 분기 함수(`Literal` 타입 힌트)로 변경해 보라.
3. 3개 이상의 분기를 가진 조건부 엣지를 설계해 보라.

In [ ]:
# Try it yourself


---
## 13.8 Send API — 동적 병렬 처리

Send API로 동일한 노드를 **다른 입력으로 여러 번 병렬 실행**:

```python
Send("노드이름", 입력값)  # 입력값은 State가 아닌 커스텀 값 가능
```

Map-Reduce 패턴: 데이터를 분할 → 병렬 처리 → Reducer로 결과 합침

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langgraph.types import Send
import operator
from typing import Union


class State(TypedDict):
    words: list[str]
    output: Annotated[list[dict[str, Union[str, int]]], operator.add]


graph_builder = StateGraph(State)


def node_one(state: State):
    print(f"I want to count {len(state['words'])} words in my state.")


# node_two는 State가 아닌 개별 word(문자열)를 받는다!
def node_two(word: str):
    return {
        "output": [
            {
                "word": word,
                "letters": len(word),
            }
        ]
    }


def dispatcher(state: State):
    return [Send("node_two", word) for word in state["words"]]


graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)

graph_builder.add_edge(START, "node_one")
graph_builder.add_conditional_edges("node_one", dispatcher, ["node_two"])
graph_builder.add_edge("node_two", END)

graph = graph_builder.compile()

result = graph.invoke(
    {
        "words": ["hello", "world", "how", "are", "you", "doing"],
    }
)

print("\nResult:")
for item in result["output"]:
    print(f"  {item['word']} -> {item['letters']} letters")

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 13.8

1. 단어 목록의 크기를 늘려 보고 결과를 관찰하라.
2. `node_two`에 `time.sleep(1)`을 추가하여 병렬 실행의 효과를 체감하라.
3. 실제 활용 사례를 생각해 보라 (여러 문서 동시 요약, 여러 소스 데이터 수집 등).

In [ ]:
# Try it yourself


---
## 13.9 Command — 노드 내부 라우팅

**Command** 객체로 **상태 업데이트 + 라우팅**을 한 번에:

```python
Command(
    goto="목적지_노드",
    update={"key": "value"},
)
```

장점: 별도의 `add_conditional_edges`나 분기 함수가 필요 없다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langgraph.types import Command
from typing import Literal


class State(TypedDict):
    transfer_reason: str


graph_builder = StateGraph(State)


def triage_node(state: State) -> Command[Literal["account_support", "tech_support"]]:
    return Command(
        goto="account_support",
        update={
            "transfer_reason": "The user wants to change password.",
        },
    )


def tech_support(state: State):
    print("tech_support running")
    return {}


def account_support(state: State):
    print("account_support running")
    return {}


graph_builder.add_node("triage_node", triage_node)
graph_builder.add_node("tech_support", tech_support)
graph_builder.add_node("account_support", account_support)

graph_builder.add_edge(START, "triage_node")
# triage_node 이후 엣지 없음 — Command가 라우팅 처리!
graph_builder.add_edge("tech_support", END)
graph_builder.add_edge("account_support", END)

graph = graph_builder.compile()
result = graph.invoke({})

print("\nResult:", result)

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 13.9

1. `triage_node`에서 조건에 따라 `tech_support`로 라우팅하도록 수정해 보라.
2. `Command`와 `add_conditional_edges` 방식의 장단점을 비교해 보라.
3. 실제 고객 지원 시스템처럼 여러 단계 라우팅을 `Command`로 구현해 보라.

In [ ]:
# Try it yourself


---
## Final Exercises — 종합 실습

### 과제 1: 카운터 그래프 (★☆☆)

4개의 노드로 구성된 선형 그래프. 상태에 `counter: int`를 두고 각 노드가 +1. 최종 값 = 4.

In [ ]:
# 과제 1: 여기에 작성


### 과제 2: 채팅 메시지 누적 (★★☆)

리듀서로 간단한 채팅 시뮬레이터:
- `user_node`: `["사용자: 안녕하세요"]`
- `assistant_node`: `["어시스턴트: 무엇을 도와드릴까요?"]`
- `user_reply_node`: `["사용자: 날씨 알려주세요"]`

In [ ]:
# 과제 2: 여기에 작성


### 과제 3: 나이별 조건부 라우팅 (★★☆)

- 18세 미만 → `minor_node` → "미성년자입니다."
- 18~64세 → `adult_node` → "성인입니다."
- 65세 이상 → `senior_node` → "경로 우대 대상입니다."

In [ ]:
# 과제 3: 여기에 작성


### 과제 4: Send API로 대문자 변환 (★★★)

문장을 단어로 분리 → 각 단어를 Send API로 병렬 대문자 변환
- 입력: `{"sentence": "hello world from langgraph"}`
- 기대: `{"results": ["HELLO", "WORLD", "FROM", "LANGGRAPH"]}`

In [ ]:
# 과제 4: 여기에 작성


### 과제 5: Command 기반 고객 상담 라우터 (★★★)

쿼리 내용에 따라 Command로 라우팅:
- "환불"/"결제" → `billing_node`
- "오류"/"버그" → `tech_node`
- 그 외 → `general_node`

In [ ]:
# 과제 5: 여기에 작성
